In [1]:
import pandas as pd

df = pd.read_csv("train.csv")
df_org = df.copy()
print(df.shape)
print(df.info())

(1460, 81)
<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   str    
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   str    
 6   Alley          91 non-null     str    
 7   LotShape       1460 non-null   str    
 8   LandContour    1460 non-null   str    
 9   Utilities      1460 non-null   str    
 10  LotConfig      1460 non-null   str    
 11  LandSlope      1460 non-null   str    
 12  Neighborhood   1460 non-null   str    
 13  Condition1     1460 non-null   str    
 14  Condition2     1460 non-null   str    
 15  BldgType       1460 non-null   str    
 16  HouseStyle     1460 non-null   str    
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [2]:
print(df.isnull().sum()[df.isnull().sum()> 0])

LotFrontage      259
Alley           1369
MasVnrType       872
MasVnrArea         8
BsmtQual          37
BsmtCond          37
BsmtExposure      38
BsmtFinType1      37
BsmtFinType2      38
Electrical         1
FireplaceQu      690
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
PoolQC          1453
Fence           1179
MiscFeature     1406
dtype: int64


In [3]:
df["Alley"] = df["Alley"].fillna("None")

In [4]:
df['MasVnrType'] = df["MasVnrType"].fillna("None")

In [5]:
df["BsmtQual"] = df['BsmtQual'].fillna("None")

In [6]:
df["BsmtCond"] = df['BsmtCond'].fillna("None")

In [7]:
df['BsmtExposure'] = df['BsmtExposure'].fillna("None")

In [8]:
df['BsmtFinType1'] = df['BsmtFinType1'].fillna("None")

In [9]:
df['BsmtFinType2'] = df['BsmtFinType2'].fillna("None")

In [10]:
df['FireplaceQu'] = df["FireplaceQu"].fillna("None")

In [11]:
cols = ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature']

df[cols] = df[cols].fillna('None')

In [12]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['SalePrice'])
y = df["SalePrice"]

X_train , X_test , y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [13]:
val = X_train['LotFrontage'].median()

X_train["LotFrontage"] = X_train['LotFrontage'].fillna(val)
X_test['LotFrontage'] = X_test['LotFrontage'].fillna(val)

In [14]:
val1 = X_train['MasVnrArea'].median()

X_train['MasVnrArea'] = X_train["MasVnrArea"].fillna(val1)
X_test['MasVnrArea'] = X_test["MasVnrArea"].fillna(val1)


In [15]:
e_val = X_train['Electrical'].mode()[0]

X_train['Electrical'] = X_train['Electrical'].fillna(e_val)


In [16]:
g_val = X_train['GarageYrBlt'].median()

X_train['GarageYrBlt'] = X_train['GarageYrBlt'].fillna(g_val)
X_test["GarageYrBlt"] = X_test['GarageYrBlt'].fillna(g_val)

In [17]:
X_train['MSSubClass'] = X_train["MSSubClass"].astype(str)
X_test['MSSubClass'] = X_test['MSSubClass'].astype(str)

In [18]:
X_train = X_train.drop(columns=['Id'])
X_test = X_test.drop(columns = ['Id'])

In [19]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

num_cols = X_train.select_dtypes(include = 'number').columns

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols] = scaler.fit_transform(X_train_scaled[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test_scaled[num_cols])

In [20]:
X_train = X_train.apply(lambda x: x.str.strip() if x.dtype == 'str' else x)
X_test = X_test.apply(lambda x: x.str.strip() if x.dtype == 'str' else x)

In [21]:
str_cols = X_train.select_dtypes(include = 'str').columns
X_train_scaled = pd.get_dummies(X_train_scaled, columns= str_cols, drop_first=True)
X_test_scaled = pd.get_dummies(X_test_scaled, columns= str_cols, drop_first=True)

X_train_scaled, X_test_scaled = X_train_scaled.align(X_test_scaled, join = 'left', axis = 1, fill_value=0)

In [26]:
X_test_scaled.isnull().sum().sum()

np.int64(0)

In [24]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

models = {
    "Linear Regression": LinearRegression(),#no params needed in continuous
    "Decision Tree": DecisionTreeRegressor(random_state=42), 
    "Random Forest": RandomForestRegressor(random_state=42)
}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    
    train_pred = model.predict(X_train_scaled)
    test_pred = model.predict(X_test_scaled)
    
    print(f"--- {name} ---")
    print(f"Train MAE: {mean_absolute_error(y_train, train_pred):.2f}  |  Test MAE: {mean_absolute_error(y_test, test_pred):.2f}")
    print(f"Train RMSE: {mean_squared_error(y_train, train_pred)**0.5:.2f}  |  Test RMSE: {mean_squared_error(y_test, test_pred)**0.5:.2f}")
    print(f"Train R2: {r2_score(y_train, train_pred):.3f}  |  Test R2: {r2_score(y_test, test_pred):.3f}")
    print()


--- Linear Regression ---
Train MAE: 12094.69  |  Test MAE: 1065917.83
Train RMSE: 18828.64  |  Test RMSE: 1076760.09
Train R2: 0.941  |  Test R2: -150.156

--- Decision Tree ---
Train MAE: 0.00  |  Test MAE: 29088.79
Train RMSE: 0.00  |  Test RMSE: 43725.79
Train R2: 1.000  |  Test R2: 0.751

--- Random Forest ---
Train MAE: 6604.15  |  Test MAE: 17962.42
Train RMSE: 11302.18  |  Test RMSE: 30343.60
Train R2: 0.979  |  Test R2: 0.880



In [ ]:
from sklearn.model_selection import cross_val_score

for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2')
    print(f"{name}: {scores.mean():.3f} (+/- {scores.std():.3f})")
    

Linear Regression: 0.471 (+/- 0.323)
Decision Tree: 0.664 (+/- 0.076)
Random Forest: 0.838 (+/- 0.048)
